# SwiftDash Food Delivery - Exploratory Data Analysis
## Operations & Customer Analytics

**Objective:** Understand customer behavior, operational efficiency, revenue drivers, and delivery performance for a food delivery platform.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

CLEAN_DIR = Path('..') / 'data' / 'cleaned'
PROC_DIR = Path('..') / 'data' / 'processed'

print('Libraries imported successfully')

In [ ]:
# Load cleaned data
customers = pd.read_csv(CLEAN_DIR / 'customers_clean.csv')
restaurants = pd.read_csv(CLEAN_DIR / 'restaurants_clean.csv')
drivers = pd.read_csv(CLEAN_DIR / 'drivers_clean.csv')
orders = pd.read_csv(CLEAN_DIR / 'orders_clean.csv')
order_items = pd.read_csv(CLEAN_DIR / 'order_items_clean.csv')
delivery_logs = pd.read_csv(CLEAN_DIR / 'delivery_logs_clean.csv')

datasets = {
    'Customers': customers, 'Restaurants': restaurants, 'Drivers': drivers,
    'Orders': orders, 'Order Items': order_items, 'Delivery Logs': delivery_logs
}
for name, df in datasets.items():
    print(f'{name:15s} -> {df.shape[0]:>8,} rows x {df.shape[1]} cols')

---
## 1. CUSTOMER ANALYSIS

In [ ]:
# Customer demographics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
customers['age'].hist(bins=30, ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Customer Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Gender distribution
gender_counts = customers['gender'].value_counts()
axes[1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#3498db', '#e74c3c', '#95a5a6'], startangle=90)
axes[1].set_title('Gender Distribution')

# Top cities
top_cities = customers['city'].value_counts().head(10)
axes[2].barh(range(len(top_cities)), top_cities.values, color='#2ecc71')
axes[2].set_yticks(range(len(top_cities)))
axes[2].set_yticklabels(top_cities.index)
axes[2].set_title('Top 10 Cities by Customer Count')
axes[2].set_xlabel('Customers')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('../screenshots/customer_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. ORDER & REVENUE ANALYSIS

In [ ]:
# Revenue trends
orders['order_date'] = pd.to_datetime(orders['order_date'])
delivered = orders[orders['order_status'].isin(['Delivered', 'Refunded'])].copy()

daily_rev = delivered.groupby('order_date')['total_amount'].sum().reset_index()
monthly_rev = delivered.set_index('order_date').resample('ME')['total_amount'].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].plot(daily_rev['order_date'], daily_rev['total_amount'], color='#3498db', alpha=0.7, linewidth=0.8)
axes[0].set_title('Daily Revenue Trend')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Revenue (INR)')

axes[1].bar(monthly_rev['order_date'].astype(str), monthly_rev['total_amount'], color='#2ecc71', width=0.7)
axes[1].set_title('Monthly Revenue')
axes[1].set_xlabel('Month')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Revenue (INR)')

plt.tight_layout()
plt.show()

In [ ]:
# Revenue by payment method
pmt_rev = delivered.groupby('payment_method').agg(
    orders=('order_id', 'count'),
    revenue=('total_amount', 'sum')
).sort_values('revenue', ascending=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

axes[0].bar(pmt_rev['payment_method'], pmt_rev['revenue'], color=colors[:len(pmt_rev)])
axes[0].set_title('Revenue by Payment Method')
axes[0].set_xlabel('Payment Method')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Revenue (INR)')

axes[1].pie(pmt_rev['revenue'], labels=pmt_rev['payment_method'], autopct='%1.1f%%',
            colors=colors[:len(pmt_rev)], startangle=90)
axes[1].set_title('Revenue Share by Payment')

plt.tight_layout()
plt.show()

## 3. DELIVERY OPERATIONS

In [ ]:
# Delivery performance
dl = delivery_logs.copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Travel time distribution
axes[0, 0].hist(dl['travel_time_mins'], bins=40, color='#3498db', edgecolor='white')
axes[0, 0].axvline(dl['travel_time_mins'].median(), color='red', linestyle='--', label=f"Median: {dl['travel_time_mins'].median():.0f} min")
axes[0, 0].set_title('Delivery Travel Time Distribution')
axes[0, 0].set_xlabel('Travel Time (mins)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# On-time rate by traffic
traffic_perf = dl.groupby('traffic_condition')['is_on_time'].mean().sort_values()
axes[0, 1].barh(traffic_perf.index, traffic_perf.values * 100, color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'])
axes[0, 1].set_title('On-Time Delivery Rate by Traffic')
axes[0, 1].set_xlabel('On-Time Rate (%)')

# Weather impact
weather_perf = dl.groupby('weather_condition')['is_on_time'].mean().sort_values()
axes[1, 0].bar(weather_perf.index, weather_perf.values * 100, color=['#3498db', '#95a5a6', '#f39c12', '#e74c3c', '#9b59b6'])
axes[1, 0].set_title('On-Time Rate by Weather')
axes[1, 0].set_xlabel('Weather')
axes[1, 0].tick_params(axis='x', rotation=30)
axes[1, 0].set_ylabel('On-Time Rate (%)')

# Delivery distance distribution
axes[1, 1].hist(dl['distance_km'], bins=40, color='#2ecc71', edgecolor='white')
axes[1, 1].set_title('Delivery Distance Distribution')
axes[1, 1].set_xlabel('Distance (km)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../screenshots/delivery_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. HOURLY & WEEKLY PATTERNS

In [ ]:
# Hourly and weekly patterns
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Orders by hour
hourly = delivered.groupby('order_hour').size()
axes[0].bar(hourly.index, hourly.values, color='#3498db')
axes[0].set_title('Orders by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Orders')
axes[0].set_xticks(range(0, 24))

# Orders by weekday
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly = delivered['weekday'].value_counts().reindex(weekday_order)
axes[1].bar(weekly.index, weekly.values, color=['#3498db']*5 + ['#e74c3c']*2)
axes[1].set_title('Orders by Day of Week')
axes[1].set_xlabel('Weekday')
axes[1].set_ylabel('Orders')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. CUISINE & RESTAURANT ANALYSIS

In [ ]:
# Merge for restaurant analysis
orders_rest = delivered.merge(restaurants[['restaurant_id', 'cuisine_type', 'name']], on='restaurant_id')

cuisine_rev = orders_rest.groupby('cuisine_type').agg(
    revenue=('total_amount', 'sum'),
    orders=('order_id', 'count')
).sort_values('revenue', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(range(len(cuisine_rev)), cuisine_rev['revenue'].values, color='#f39c12')
axes[0].set_yticks(range(len(cuisine_rev)))
axes[0].set_yticklabels(cuisine_rev.index)
axes[0].set_title('Top 10 Cuisines by Revenue')
axes[0].set_xlabel('Revenue (INR)')
axes[0].invert_yaxis()

axes[1].barh(range(len(cuisine_rev)), cuisine_rev['orders'].values, color='#e74c3c')
axes[1].set_yticks(range(len(cuisine_rev)))
axes[1].set_yticklabels(cuisine_rev.index)
axes[1].set_title('Top 10 Cuisines by Order Volume')
axes[1].set_xlabel('Orders')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. CUSTOMER SEGMENTATION (RFM)

In [ ]:
customer_features = pd.read_csv(PROC_DIR / 'customer_features.csv')

segment_counts = customer_features['customer_segment'].value_counts()
segment_revenue = customer_features.groupby('customer_segment')['monetary'].sum() / 1e6

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_seg = {'Platinum': '#2c3e50', 'Gold': '#f39c12', 'Silver': '#3498db',
              'At Risk': '#e67e22', 'Churned': '#e74c3c', 'New': '#95a5a6'}

axes[0].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%',
            colors=[colors_seg.get(s, '#333') for s in segment_counts.index], startangle=90)
axes[0].set_title('Customer Segment Distribution')

axes[1].bar(segment_revenue.index, segment_revenue.values,
            color=[colors_seg.get(s, '#333') for s in segment_revenue.index])
axes[1].set_title('Revenue by Customer Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Revenue (Millions INR)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../screenshots/customer_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. ORDER ITEMS ANALYSIS

In [ ]:
# Top items
items = order_items.groupby('item_name').agg(
    orders=('order_id', 'nunique'),
    quantity=('quantity', 'sum'),
    revenue=('line_total', 'sum')
).sort_values('orders', ascending=False).head(15)

plt.figure(figsize=(14, 6))
plt.barh(range(len(items)), items['orders'].values, color='#9b59b6')
plt.yticks(range(len(items)), items.index)
plt.title('Top 15 Most Ordered Items')
plt.xlabel('Number of Orders')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../screenshots/top_items.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. CORRELATION HEATMAP

In [ ]:
# Numerical correlations
num_cols = ['order_amount', 'delivery_fee', 'discount', 'tax', 'platform_fee',
            'surge_multiplier', 'total_amount']
corr = delivered[num_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=1)
plt.title('Order Variables Correlation Matrix')
plt.tight_layout()
plt.savefig('../screenshots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. KEY INSIGHTS SUMMARY

In [ ]:
print('='*60)
print('EDA KEY FINDINGS SUMMARY')
print('='*60)

total_rev = delivered['total_amount'].sum()
total_orders = delivered['order_id'].nunique()
aov = delivered['total_amount'].mean()
cancellation_rate = (orders['order_status'] == 'Cancelled').mean() * 100
on_time_rate = delivery_logs['is_on_time'].mean() * 100
avg_delivery = delivery_logs['travel_time_mins'].mean()
top_payment = delivered.groupby('payment_method')['total_amount'].sum().idxmax()
peak_hour = delivered.groupby('order_hour').size().idxmax()
top_cuisine = orders_rest.groupby('cuisine_type')['total_amount'].sum().idxmax()

print(f'\n1. Total Revenue (Delivered Orders): INR {total_rev:,.2f}')
print(f'2. Total Delivered Orders: {total_orders:,}')
print(f'3. Average Order Value: INR {aov:,.2f}')
print(f'4. Overall Cancellation Rate: {cancellation_rate:.2f}%')
print(f'5. On-Time Delivery Rate: {on_time_rate:.2f}%')
print(f'6. Average Delivery Time: {avg_delivery:.1f} minutes')
print(f'7. Most Popular Payment Method: {top_payment}')
print(f'8. Peak Order Hour: {peak_hour}:00')
print(f'9. Highest Revenue Cuisine: {top_cuisine}')

print('\n' + '='*60)
print('EDA Complete - See screenshots/ for visual exports')